In [3]:

# ASSIGNMENT 2: THE DIRTY LEDGER

import pandas as pd


# STEP 1: LOAD THE DATA

df = pd.read_csv(r"C:\Users\dell\Downloads\teaching_prices_messy.csv",
                 dtype=str, keep_default_na=False)

original_columns = list(df.columns)
df["csv_row"] = range(2, len(df) + 2)

print("ORIGINAL DATA")
print(df.head())

print("Rows:", len(df))
print("Columns:", original_columns)
print("Data types:")
print(df[original_columns].dtypes)


# STEP 2: FIND PROBLEMS IN THE DATA

print("\nDATA QUALITY CHECKS")

# Check missing prices
print("Missing prices:")
print((df["price"] == "").sum())

# Check duplicates
print("Duplicate rows:")
print(df.duplicated(subset=original_columns).sum())

# Check item names
print("Item names:")
print(df["item"].value_counts())

# Check missing price records
print("\nRows with missing prices:")
print(df[df["price"] == ""])

# Check prices containing commas
print("\nPrices containing commas:")
print(df[df["price"].str.contains(",")])

# Convert prices temporarily for checking
numbers = pd.to_numeric(
    df["price"].str.replace(",", "", regex=False),
    errors="coerce"
)

# Check unusually high prices
print("\nPrices above 500:")
print(df[numbers > 500])

# Check incorrect item spelling
print("\nItem named oil b:")
print(df[df["item"] == "oil b"])

# Examine all records on that date
print("\nRecords on 2026-02-05:")
print(df[df["date"] == "2026-02-05"])

# Show duplicate records
print("\nRepeated records:")
print(df[df.duplicated(
    subset=original_columns, keep=False
)])


# STEP 3: WRITE PLANNED CLEANING DECISIONS

plan = pd.DataFrame({
    "Problem": [
        "Missing price",
        "Price 1,250",
        "Price 9999",
        "Incorrect item name",
        "Duplicate row"
    ],
    "Decision": [
        "Keep and flag",
        "Convert and flag",
        "Keep original and flag",
        "Use inferred name and flag",
        "Remove extra copy"
    ],
    "Reason": [
        "Correct price is unknown",
        "Price is unusually high",
        "No confirmed correct price",
        "Oil B already exists on the same date",
        "All original values are identical"
    ]
})

print("\nPLANNED DECISIONS")
print(plan.to_string(index=False))


# STEP 4: CLEAN THE DATA

# Make a copy to protect original data
clean = df.copy()

# Remove exact duplicate records
clean = clean.drop_duplicates(
    subset=original_columns
).copy()

# Keep the original item name
# Create a separate cleaned item column
clean["item_clean"] = clean["item"]

# Correct the suspected label
# This is an inference, not a confirmed correction
clean.loc[
    clean["item"] == "oil b", "item_clean"
] = "Staple A"

# Convert prices into numbers
# Original price column stays unchanged
clean["price_numeric"] = clean["price"].str.replace(
    ",", "", regex=False
)

clean["price_numeric"] = pd.to_numeric(
    clean["price_numeric"],
    errors="coerce"
)

# Create a price column for analysis
clean["price_for_analysis"] = clean["price_numeric"]

# Flag unusually high prices
# 500 is a review threshold for this dataset only
clean.loc[
    clean["price_numeric"] > 500,
    "price_for_analysis"
] = float("nan")

# Add notes to explain decisions
clean["note"] = "OK"

clean.loc[
    clean["price_numeric"].isna(),
    "note"
] = "Missing price - check source"

clean.loc[
    clean["price_numeric"] > 500,
    "note"
] = "High price - verify"

clean.loc[
    clean["item"] == "oil b",
    "note"
] = "Item name inferred - verify"

print("\nCLEANED DATA")
print(clean.head())


# STEP 5: CHECK THE CHANGED ROWS

print("\nROWS NEEDING REVIEW")

print(clean[clean["note"] != "OK"][
    ["csv_row", "date", "item", "item_clean",
     "city", "price", "price_numeric",
     "price_for_analysis", "note"]
].to_string(index=False))

print("\nNumber of original rows:")
print(len(df))

print("Number of cleaned rows:")
print(len(clean))

print("Rows removed:")
print(len(df) - len(clean))


# STEP 6: BEFORE AND AFTER COMPARISON

comparison = pd.DataFrame({
    "Check": [
        "Total rows",
        "Missing or flagged prices",
        "Different item labels",
        "Duplicate complete rows",
        "Duplicate date-item-city keys",
        "Maximum analysis price"
    ],
    "Before": [
        len(df),
        numbers.isna().sum(),
        df["item"].nunique(),
        df.duplicated(subset=original_columns).sum(),
        df.duplicated(
            subset=["date", "item", "city"]
        ).sum(),
        numbers.max()
    ],
    "After": [
        len(clean),
        clean["price_for_analysis"].isna().sum(),
        clean["item_clean"].nunique(),
        clean.duplicated(
            subset=original_columns
        ).sum(),
        clean.duplicated(
            subset=["date", "item_clean", "city"]
        ).sum(),
        clean["price_for_analysis"].max()
    ]
})

print("\nBEFORE AND AFTER COMPARISON")
print(comparison.to_string(index=False))

print("\nBEFORE CLEANING STATISTICS")
print(numbers.describe())

print("\nAFTER CLEANING STATISTICS")
print(clean["price_for_analysis"].describe())


# STEP 7: AI CODE COURT
# Test suggestions before deciding to accept them.

# Trial 1: Directly convert price to numbers
trial = df.copy()

trial["price"] = pd.to_numeric(
    trial["price"], errors="coerce"
)

print("\nAI TRIAL 1: DIRECT CONVERSION")

print("Missing after direct conversion:")
print(trial["price"].isna().sum())

print("Missing after removing commas first:")
print(numbers.isna().sum())


# Trial 2: Automatically capitalize item names
trial2 = df.copy()

trial2["item"] = trial2["item"].str.title()

print("\nAI TRIAL 2: CAPITALIZE ITEM NAMES")

print("Duplicate date-item-city keys:")
print(trial2.duplicated(
    subset=["date", "item", "city"]
).sum())


# Record decisions about AI suggestions

court = pd.DataFrame({
    "AI suggestion": [
        "Remove exact duplicates",
        "Convert price directly",
        "Capitalize all item names",
        "Delete missing-price rows",
        "Fill missing price with median",
        "Replace large prices with 500",
        "Change oil b to Staple A"
    ],
    "Verdict": [
        "KEEP",
        "MODIFY",
        "REJECT",
        "REJECT",
        "REJECT",
        "REJECT",
        "MODIFY"
    ],
    "Reason": [
        "One extra copy is removed",
        "Comma price becomes missing",
        "Creates wrong duplicate item",
        "Valid record may be lost",
        "Median is only a guess",
        "500 is an invented replacement",
        "Likely correct but needs verification"
    ]
})

print("\nAI CODE COURT RESULTS")
print(court.to_string(index=False))


# STEP 8: FINAL DATA CHECKS

print("\nFINAL CHECKS")

print("Rows:", len(clean))

print("Missing or flagged prices:")
print(clean["price_for_analysis"].isna().sum())

print("Duplicate date-item-city records:")
print(clean.duplicated(
    subset=["date", "item_clean", "city"]
).sum())

print("Item counts:")
print(clean["item_clean"].value_counts())

# Check expected results
assert len(clean) == 96

assert clean["price_for_analysis"].isna().sum() == 3

assert clean.duplicated(
    subset=["date", "item_clean", "city"]
).sum() == 0

print("\nALL CHECKS PASSED")


# STEP 9: CREATE CLEANING LOG

log = plan.copy()

print("\nCLEANING LOG")
print(log.to_string(index=False))

ORIGINAL DATA
         date      item    city  price                       unit  \
0  2026-01-01  Staple A  City 1  101.0  PKR per kg (illustrative)   
1  2026-01-01  Staple A  City 2  107.0  PKR per kg (illustrative)   
2  2026-01-01     Oil B  City 1  298.0  PKR per kg (illustrative)   
3  2026-01-01     Oil B  City 2  304.0  PKR per kg (illustrative)   
4  2026-01-01   Pulse C  City 1  200.0  PKR per kg (illustrative)   

                                          data_label  csv_row  
0  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...        2  
1  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...        3  
2  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...        4  
3  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...        5  
4  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...        6  
Rows: 97
Columns: ['date', 'item', 'city', 'price', 'unit', 'data_label']
Data types:
date          object
item          object
city          object
price         object
unit          obj